# Experiment: two-stage cascade (detector + categorizer)
Minimal copy of the Task 1-5 code of `Deliverable.ipynb`. Same `BiLSTMClassifier` as the Baseline (1 layer, max pooling), used twice:

1. **Stage 1, detector:** 1 logit, "is the clause unfair?". Trained on the whole train set.
2. **Stage 2, categorizer:** 5 logits, "which categories?". Trained only on the positive train clauses.

Same strategy as Task 5: `A` x4 on train, `pos_weight` computed on each stage's train data, best epoch on validation, thresholds tuned on validation only. Two ways to combine the stages:

- **gate:** stage 2 counts only where stage 1 is above its threshold `t1`;
- **product:** `P(c) = P(unfair) * P(c | unfair)`, with one threshold per category.

Runs go to `Experiments/results/task5_runs_cascade.pkl`, weights to `Experiments/results/weights_cascade/`. Only validation is printed. Not part of the submission.

In [1]:
ROOT = Path.cwd().parents[1]  # this notebook lives in Experiments/notebooks
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

data_folder = ROOT / 'Data' / 'ToS-100'
keep = ['document_ID', 'text', 'A', 'CH', 'CR', 'LTD', 'TER']
training_set = pd.read_csv(data_folder / 'train.csv')[keep]
test_set = pd.read_csv(data_folder / 'test.csv')[keep]
val_set = pd.read_csv(data_folder / 'validation.csv')[keep]

label_cols = ['A', 'CH', 'CR', 'LTD', 'TER']
y_val = val_set[label_cols].to_numpy(dtype=np.float32)
y_test = test_set[label_cols].to_numpy(dtype=np.float32)

In [2]:
import nltk
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

ENUMERATOR = r"(?:\d+|[a-z]|[ivx]+)"
NUMBERING_RE = re.compile(
    rf"^(?:-l[rs]b- {ENUMERATOR} -r[rs]b-"
    rf"|{ENUMERATOR} -r[rs]b-"
    rf"|\d+(?:\.\d+)+\.?"
    rf"|\d+ [.°]"
    rf"|[•·●*-])\s+"
)
TOKEN_MAP = {
    '-lrb-': '(', '-rrb-': ')', '-lsb-': '[', '-rsb-': ']', '-lcb-': '{', '-rcb-': '}',
    '``': '"', "''": '"', '`': "'",
}
CLITICS = {"'s", "'re", "'ve", "'ll", "'d", "'m", "n't"}
SPECIAL_CHARS_RE = re.compile(r"[^a-z0-9\s.,;:!?()\[\]{}\"'/$%&@-]")

lemmatizer = WordNetLemmatizer()


def get_wordnet_key(tag: str) -> str | None:
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return None


def strip_numbering(text: str) -> str:
    while (stripped := NUMBERING_RE.sub('', text, count=1)) != text:
        text = stripped
    return text


def lemmatize(tokens: list[str]) -> list[str]:
    lemmas = []
    for i, (word, tag) in enumerate(pos_tag(tokens)):
        key = get_wordnet_key(tag)
        before_negation = i + 1 < len(tokens) and tokens[i + 1] == "n't"
        lemmas.append(lemmatizer.lemmatize(word, key) if key and not before_negation else word)
    return lemmas


def reattach_clitics(tokens: list[str]) -> list[str]:
    merged = []
    for token in tokens:
        if token in CLITICS and merged:
            merged[-1] += token
        else:
            merged.append(token)
    return merged


def clean_text(text: str) -> str:
    text = strip_numbering(text)
    tokens = [TOKEN_MAP.get(token, token) for token in text.split()]
    tokens = reattach_clitics(lemmatize(tokens))
    text = SPECIAL_CHARS_RE.sub(' ', ' '.join(tokens))
    return ' '.join(text.split())


for df in (training_set, val_set, test_set):
    df['clean_text'] = df['text'].apply(clean_text)

In [3]:
import gensim.downloader as gloader

glove = gloader.load('glove-wiki-gigaword-100')

PAD, UNK = '<PAD>', '<UNK>'
PAD_IDX, UNK_IDX = 0, 1

train_counts = Counter(token for text in training_set['clean_text'] for token in text.split())
train_oov = [token for token in train_counts if token not in glove.key_to_index]

idx_to_word = [PAD, UNK] + glove.index_to_key + train_oov
word_to_idx = {word: idx for idx, word in enumerate(idx_to_word)}

def compose_vector(token: str) -> np.ndarray | None:
    clitic = next((c for c in CLITICS if token.endswith(c) and token != c), None)
    stem = token[:-len(clitic)] if clitic else token
    parts = re.split(r'[-/]', stem) + ([clitic] if clitic else [])
    vectors = [glove[part] for part in parts if part in glove.key_to_index]
    return np.mean(vectors, axis=0) if vectors else None


rng = np.random.default_rng(42)
embedding_dim = glove.vector_size

embedding_matrix = np.zeros((len(idx_to_word), embedding_dim), dtype=np.float32)
embedding_matrix[UNK_IDX] = rng.uniform(-0.05, 0.05, embedding_dim)
embedding_matrix[2:2 + len(glove.index_to_key)] = glove.vectors

composed = []
for token in train_oov:
    vector = compose_vector(token)
    if vector is None:
        vector = rng.uniform(-0.05, 0.05, embedding_dim)
    else:
        composed.append(token)
    embedding_matrix[word_to_idx[token]] = vector

def encode(text: str) -> list[int]:
    return [word_to_idx.get(token, UNK_IDX) for token in text.split()]


for df in (training_set, val_set, test_set):
    df['token_ids'] = df['clean_text'].apply(encode)

In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

MAX_LEN = 150


class ClauseDataset(Dataset):
    def __init__(self, token_ids: pd.Series, labels: np.ndarray):
        self.sequences = [torch.tensor((ids or [UNK_IDX])[:MAX_LEN]) for ids in token_ids]
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    return padded, lengths, torch.stack(labels)


import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix: np.ndarray, hidden_size: int, num_layers: int, n_labels: int = len(label_cols),
                 freeze_embedding: bool = False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=freeze_embedding, padding_idx=PAD_IDX
        )
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.dense = nn.Linear(2 * hidden_size, n_labels)

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = pack_padded_sequence(self.embedding(token_ids), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        outputs, _ = self.lstm(packed)
        states, _ = pad_packed_sequence(outputs, batch_first=True)
        padding = torch.arange(states.size(1), device=states.device) >= lengths.to(states.device).unsqueeze(1)
        return self.dense(states.masked_fill(padding.unsqueeze(-1), float('-inf')).max(dim=1).values)

In [5]:
HIDDEN_SIZE = 128
SEEDS = [42, 1337, 2026]
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

A_FACTOR = 4

a_clauses = training_set[training_set['A'] == 1]
train_resampled = pd.concat([training_set] + [a_clauses] * (A_FACTOR - 1), ignore_index=True)
y_resampled = train_resampled[label_cols].to_numpy(dtype=np.float32)

y1_train = (y_resampled.sum(axis=1, keepdims=True) > 0).astype(np.float32)
y1_val = (y_val.sum(axis=1, keepdims=True) > 0).astype(np.float32)
val_pos = y1_val[:, 0] == 1
stage2_mask = y1_train[:, 0] == 1


def compute_pos_weight(y: np.ndarray) -> torch.Tensor:
    n_pos = y.sum(axis=0)
    return torch.tensor((len(y) - n_pos) / n_pos)


val_loader = DataLoader(ClauseDataset(val_set['token_ids'], y_val), batch_size=256, collate_fn=collate)
test_loader = DataLoader(ClauseDataset(test_set['token_ids'], y_test), batch_size=256, collate_fn=collate)

THRESHOLD_GRID = np.arange(0.05, 0.96, 0.05)


def compute_metrics(y_true: np.ndarray, probs: np.ndarray, thresholds: float | np.ndarray = 0.5) -> dict:
    y_pred = (probs >= thresholds).astype(int)
    per_category = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {
        'macro F1': per_category.mean(),
        'macro precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        **{f'F1 {label}': score for label, score in zip(label_cols, per_category)},
    }


def tune_thresholds(y_true: np.ndarray, probs: np.ndarray) -> np.ndarray:
    thresholds = []
    for c in range(y_true.shape[1]):
        scores = [f1_score(y_true[:, c], probs[:, c] >= t, zero_division=0) for t in THRESHOLD_GRID]
        thresholds.append(THRESHOLD_GRID[np.argmax(scores)])
    return np.array(thresholds)


STAGES = {
    'stage1': {
        'dataset': ClauseDataset(train_resampled['token_ids'], y1_train),
        'n_labels': 1,
        'pos_weight': compute_pos_weight(y1_train),
        'val_score': lambda probs: f1_score(y1_val[:, 0], probs[:, 0] >= 0.5, zero_division=0),
    },
    'stage2': {
        'dataset': ClauseDataset(train_resampled['token_ids'][stage2_mask], y_resampled[stage2_mask]),
        'n_labels': len(label_cols),
        'pos_weight': compute_pos_weight(y_resampled[stage2_mask]),
        'val_score': lambda probs: compute_metrics(y_val[val_pos], probs[val_pos])['macro F1'],
    },
}
for name, stage in STAGES.items():
    print(f"{name}: {len(stage['dataset'])} train clauses, pos_weight {stage['pos_weight'].numpy().round(2)}")


@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    return np.concatenate([torch.sigmoid(model(ids.to(DEVICE), lengths)).cpu().numpy() for ids, lengths, _ in loader])


def train_stage(name: str, seed: int) -> dict:
    stage = STAGES[name]
    set_seed(seed)
    train_loader = DataLoader(stage['dataset'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                              generator=torch.Generator().manual_seed(seed))

    model = BiLSTMClassifier(embedding_matrix, HIDDEN_SIZE, 1, n_labels=stage['n_labels']).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=stage['pos_weight'].to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_score, best_epoch, best_state = -1.0, 0, None
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for ids, lengths, labels in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(ids.to(DEVICE), lengths), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(labels)

        score = stage['val_score'](predict_proba(model, val_loader))
        history.append({'epoch': epoch, 'train_loss': epoch_loss / len(stage['dataset']), 'val_score': score})
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    return {
        'seed': seed,
        'best_epoch': best_epoch,
        'best_val_score': best_score,
        'history': history,
        'val_probs': predict_proba(model, val_loader),
        'test_probs': predict_proba(model, test_loader),
        'state_dict': best_state,
    }

stage1: 16062 train clauses, pos_weight [10.09]
stage2: 1448 train clauses, pos_weight [3.83 4.4  7.78 1.87 3.55]


In [6]:
TRAIN_MODELS = False
RESULTS_DIR = ROOT / 'Experiments' / 'results'
RUNS_FILE = RESULTS_DIR / 'task5_runs_cascade.pkl'
WEIGHTS_DIR = RESULTS_DIR / 'weights_cascade'

if TRAIN_MODELS:
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    runs = {'stage1': [], 'stage2': []}
    for name in runs:
        for seed in SEEDS:
            start = time.time()
            run = train_stage(name, seed)
            torch.save(run.pop('state_dict'), WEIGHTS_DIR / f'{name}_seed{seed}.pt')
            runs[name].append(run)
            print(f"{name} | seed {seed} | best epoch {run['best_epoch']:2d} | "
                  f"val score {run['best_val_score']:.4f} | {time.time() - start:.0f} s")
    pd.to_pickle(runs, RUNS_FILE)
else:
    runs = pd.read_pickle(RUNS_FILE)

stage1 | seed 42 | best epoch  5 | val score 0.6417 | 220 s


stage1 | seed 1337 | best epoch  5 | val score 0.6205 | 220 s


stage1 | seed 2026 | best epoch  8 | val score 0.6266 | 216 s


stage2 | seed 42 | best epoch 18 | val score 0.8731 | 25 s


stage2 | seed 1337 | best epoch 16 | val score 0.8663 | 26 s


stage2 | seed 2026 | best epoch 14 | val score 0.8749 | 26 s


In [7]:
def tune_gate(p1: np.ndarray) -> float:
    scores = [f1_score(y1_val[:, 0], p1[:, 0] >= t, zero_division=0) for t in THRESHOLD_GRID]
    return THRESHOLD_GRID[np.argmax(scores)]


rows, cascade = [], []
for r1, r2 in zip(runs['stage1'], runs['stage2']):
    p1, p2 = r1['val_probs'], r2['val_probs']
    t1 = tune_gate(p1)
    gated = (p1 >= t1) * p2
    product = p1 * p2
    t_gate, t_product = tune_thresholds(y_val, gated), tune_thresholds(y_val, product)
    cascade.append({'seed': r1['seed'], 't1': t1, 't_gate': t_gate, 't_product': t_product})
    rows += [
        {'seed': r1['seed'], 'setting': 'gate, all 0.5', **compute_metrics(y_val, (p1 >= 0.5) * p2)},
        {'seed': r1['seed'], 'setting': 'gate, tuned', **compute_metrics(y_val, gated, t_gate)},
        {'seed': r1['seed'], 'setting': 'product, 0.5', **compute_metrics(y_val, product)},
        {'seed': r1['seed'], 'setting': 'product, tuned', **compute_metrics(y_val, product, t_product)},
    ]

reference = pd.read_pickle(RESULTS_DIR / 'task5_runs.pkl')['Baseline']
for run in reference:
    rows += [
        {'seed': run['seed'], 'setting': 'Task 5 Baseline, 0.5', **compute_metrics(y_val, run['val_probs'])},
        {'seed': run['seed'], 'setting': 'Task 5 Baseline, tuned', **compute_metrics(y_val, run['val_probs'], run['thresholds'])},
    ]

val_table = pd.DataFrame(rows).drop(columns='seed')
print('Validation, mean over 3 seeds:')
print(val_table.groupby('setting').mean().round(3).to_string())
print()
print('Validation macro F1, std over 3 seeds:')
print(val_table.groupby('setting')['macro F1'].std().round(4).to_string())

Validation, mean over 3 seeds:
                        macro F1  macro precision  macro recall   F1 A  F1 CH  F1 CR  F1 LTD  F1 TER
setting                                                                                             
Task 5 Baseline, 0.5       0.546            0.495         0.620  0.401  0.556  0.562   0.626   0.582
Task 5 Baseline, tuned     0.593            0.615         0.596  0.471  0.594  0.643   0.644   0.616
gate, all 0.5              0.555            0.499         0.640  0.396  0.556  0.651   0.576   0.594
gate, tuned                0.589            0.597         0.592  0.396  0.635  0.684   0.624   0.607
product, 0.5               0.567            0.522         0.635  0.396  0.586  0.662   0.593   0.599
product, tuned             0.594            0.641         0.566  0.418  0.636  0.678   0.629   0.608

Validation macro F1, std over 3 seeds:
setting
Task 5 Baseline, 0.5      0.0060
Task 5 Baseline, tuned    0.0067
gate, all 0.5             0.0132
gate, tuned   

In [8]:
for r1, r2, c in zip(runs['stage1'], runs['stage2'], cascade):
    p1 = r1['val_probs'][:, 0]
    t1 = c['t1']
    print(f"seed {r1['seed']}: stage 1 best epoch {r1['best_epoch']}, binary F1 {r1['best_val_score']:.3f} at 0.5, "
          f"t1 {t1:.2f}: precision {precision_score(y1_val[:, 0], p1 >= t1):.3f}, "
          f"recall {recall_score(y1_val[:, 0], p1 >= t1):.3f}")
    print(f"         stage 2 best epoch {r2['best_epoch']}, macro F1 on true positives {r2['best_val_score']:.3f}")
    print(f"         thresholds gate {c['t_gate'].round(2).tolist()}, product {c['t_product'].round(2).tolist()}")

seed 42: stage 1 best epoch 5, binary F1 0.642 at 0.5, t1 0.70: precision 0.643, recall 0.670
         stage 2 best epoch 18, macro F1 on true positives 0.873
         thresholds gate [0.15, 0.8, 0.45, 0.95, 0.5], product [0.5, 0.7, 0.8, 0.85, 0.5]
seed 1337: stage 1 best epoch 5, binary F1 0.621 at 0.5, t1 0.55: precision 0.579, recall 0.689
         stage 2 best epoch 16, macro F1 on true positives 0.866
         thresholds gate [0.2, 0.95, 0.5, 0.95, 0.7], product [0.95, 0.7, 0.5, 0.8, 0.55]
seed 2026: stage 1 best epoch 8, binary F1 0.627 at 0.5, t1 0.80: precision 0.650, recall 0.656
         stage 2 best epoch 14, macro F1 on true positives 0.875
         thresholds gate [0.75, 0.95, 0.4, 0.95, 0.5], product [0.75, 0.9, 0.7, 0.9, 0.65]
